INTRODUZIONE AI MODELLI NEURALI PER NLP (RNN, LSTM, GRU)

L'idea di fondo dei modelli neurali per NLP è questa: invece di rappresentare il testo solo con i conteggi come Bow o TF-IDF, si cerca di elaborare il testo come sequenza ordinata di token.
Una frase come:
"il cliente ordina il prodotto"
non viene trattata solo come insieme di parole, ma come sequenza 
il -> cliente -> ordina -> il -> prodotto
Questo è importante perchè nell NLP l'ordine conta.
Esempio
"il cane morde l'uomo" e "l'uomo morde il cane" contengono le stessa parole, ma hanno significati diversi.
I modelli RNN, LSTM, GRU sono nati proprio per lavorare bene su dati sequenziali.
La struttura concettuale è:
testo -> tokenizzazione -> token ID -> embedding -> RNN/LSTM/GRU -> rappresentazione delle sequenza -> output

L'output dipende dal problema.
recensione -> LSTM -> positiva/negativa
sequenza di parole -> RNN -> previsione della parole successiva.

Entriamo pertanto nel vivo del Deep Learningn per il linguaggio
Sappiamo trasformare le parole in numeri, ma il linguaggio non è una lista della spesa, è un flusso, un flusso con memoria..
Diamo alle macchine la capacità di ricordare, analizzando le RNN, LSTM e le GRU

Necessità di memoria
Perchè una rete densa tradizionale fallisce con il testo?
perchè tratta ogni parola come un atomo isolato.
Il significato risiede nella correlazione temporale, dobbiamo passare da un'elaborazione statica ad una dinamica, dove l'ordine è sovrano.

Ma come facciamo tecnicamente a fare in modo che la rete non scordi ciò che ha appena letto?
il segresto sta nel concetto di hidden state

Elementi della persistenza informativa
Come i dati fluiscono nel tempo
- Dipendenza temporale: è la capacità di un sistema di alterare l'output corrente in base agli input ricevuti nei passi temporali precedenti. Mentra legge la frase, la rete scrive sul taccuino del passato. 
- Stato Nascosto (Hidden State): è un vettore che funge da memoria di lavoro, aggionato costantemente a ogni nuovo token della sequenza.
- Ricorsione: è il processo per cui l'output di un neurone al tempo 't' diventa parte dell'input per lo stesso neurone al tempo t+1
- Contesto: è l'accumulo di informazioni semantiche necessarie per risolvere ambiguità linguistiche, come i riferimenti pronominali.

E' così che costruiamo il contesto.

Vediamo ora la meccanica che permette al taccuino di aggiorarsi
Meccanismi di memoria neurale.
Le reti classiche dense non hanno una nozione intrinseca di ordine. Se scambiano l'ordine delle parole in un vettore 'Bag of Words' l'input per una rete densa non cambia, ma il significato si (limiti reti standard).
In una RNN, la computazione dello stato attuale è una funzione del termine corrente e dello stato precedente permettendo la propagazione dell'informazione lungo la catena (il loop ricorrente).E' come una staffetta, ogni corridoere passa il testimone al successivo. metematicamente lo stato attuale è una funzione dello stsato attuale e dello stato precedente.
Possiamo visualizzare una RNN come una serie di coppie identiche della stessa rete, ognuna della quali passi un messaggio alla sucessiva (unfolding nel tempo).

Entriamo più nel dettaglio matematico di questo vettore di stato

Dal segnale al vettore di stato
Matematica della persistenza
Il vettore 'h' non è solo un output, ma una rappresentazione compressa della storia della sequenza fino a quel momento.
La sfida pricipale è decidere quanta importanza dare al nuovo input rispetto alla memoria accumulata, un bilanciamenteo regolato dai pesi addestrabili della rete.
La vera sfida della rete durante l'addestramento è imparare i pesi corretti dando il giusto peso alla parola attuale a quanto peso dare a tutto ià che ha letto prima. Troppa memoria ed il modello si confonde, troppa enfasi sul presente ed il modello perde il contesto

Ma prima dobbiamo risolvere un problema pratico.
Le frasi non sono ttute lunghe uguali.
Le immagini se sono grandi le rimpiccioliamo se sono piccole le ingrandiamo. con il testo non è cos' semplice.

Le nostre GPU hanno però bisogno di geometrica, di matrici regolari per calcolare in parallelo.
Dobbiamo trovare un modo per far entrare la fluidità di un linguaggio in questi tensori a forma fissa.
Questo ci porta tra le tecniche di norrmalizzazione delle sequenze.

Normalizzazione delle sequenze
Qui entrano in gioco 3 tecniche
1) Padding: aggiunta di token nulli (solitamente zeri) per portare tutte le frasi di un batch alla stessa lunghezza massima
2) Truncation: taglio dei token eccedenti per le frasi che superano una soglia massima prestabilita di lunghezza
3) Masking: il macking è fondamentale è il meccanismo che istruisce la rete a ignorare i token di padding durante il calcolo della loss e dell'aggirnamento dei pesi

La Dynamic Unrolling è la capacità di alcuni framework di gestire sequenze di lunghezza diversa senza padding eccessivo, sebbene meno efficiente su larga scala.

Ma dove mettiamo questi zeri? all'inizio o alla fine?
La scelta tra pre-padding e post-padding non è solo estetica e può influenzare drasticamente lo stasto finale della memoria.
Senza mascking, la rete cercherebbe di 'imparare' il significato degli zeri, degradando la qualità delle rappresentazioni vettoriali

Spesso si preferisce il pre-padding, ed il mascking agisce come filtro che nulla il contributo dei dati silenti

Per non sprecare potenza di calcolo, usiamo il batching intelligente. Invece di rieppire tutto di zero, raggruppiamo frasi di lunghezza simile, in Keras la libreria 'pad_sequences' automatizza tutto questo, per trasformare liste di token in matrici pronte per il training.

Ora che i dati sono pronti, affrontiamo il grande limite delle RNN semplici
Le RNN classiche hanno un problema di memoria a breve termine.
Le RNN hanno lo stesso problema, in pochi passaggi il segnale iniziale svanisce.
Matematicamente questo è il collo di bottiglia della backpropagation attraverso il tempo, che porta i gradienti ad annullarsi o esplodere.

Ma perchè il segnale svanisce?
Durante l'addestramento, il gradiente diventa infinitesimale, impedendo ai pesi iniziali di aggiornarsi. L'informazione letteralmente evapora.
Le RNN classiche non riescono a ricordare informazioni tra loro distanti, per fortuna sono nate le architettura a Gate logici, o cancelli, (input, forget, output) per decidere attivamente cosa memorizzare e cosa dimenticare.

L'evoluzione è stata cambiare l'anatomia del neurone: LSTM e GRU
* LSTM (Long Short-Term Memory): introduce la 'Cell State', un'autostrada di informazioni che attraversa tutta la catena con minime interazioni lineari preservando il gradiente. I cancelli (input, forget, output) decisono cosa far entrare e cosa cancellare.
- GRU (Gated Recurrent Unit): una versione semplificata di LSTM che fonde i cancelli, offrendo prestazioni simili ma con una velocità computazionale superiore.

- Mentre la RNN ha un solo stato, la LSTM gestisce due vettori distinti per separare la memoria a lungo termine da quella a breve termine.

Queste reti hanno dominato NLP per un decennio prima dell'arrivo dei transformer.

Ma come si comportano queste celle nella pratica del codice

Quando scrivere SimpleRNN in keras, sappiate che state usando un modello che farà fatica su testi più lunghi di 10 o 15 parole.
Passando a LSTM noterete subito un salto di accuratezza. Un dettaglio tecnico, usiamo quasi sempre la funzione di attivazione'tanh' per mantenere i valori dello stasto entro un range controllato (tra -1 e 1) ed evitare divergenze numeriche.

In [1]:
"""
================================================================================
MODELLI NEURALI PER SEQUENZE (NLP)
================================================================================

Questo script spiega come le macchine imparano a leggere.
Vedremo come trasformare parole in numeri, come gestire frasi di lunghezze diverse
e perché le reti LSTM sono migliori delle semplici RNN per "ricordare" il contesto.

ARCHITETTURA LOGICA:
1. TESTO GREGGIO -> 2. VETTORIZZAZIONE (numeri) -> 3. EMBEDDING (concetti) -> 
4. LSTM (memoria) -> 5. DENSE (decisione finale).
"""

import os

# --- PASSO 0: CONFIGURAZIONE MOTORE ---
# Keras 3 è un'interfaccia universale. Qui diciamo a Keras di usare PyTorch 
# come "motore aritmetico" (backend) per eseguire i calcoli.
os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers
import numpy as np

def crea_dataset_didattico():
    """
    FASE 1: PREPARAZIONE MATERIA PRIMA.
    Simuliamo delle recensioni o frasi che la rete dovrà analizzare.
    
    PERCHÉ QUESTE FRASI? 
    Hanno lunghezze diverse (da 4 a 10 parole). Questo serve a testare il PADDING.
    """
    testi = [
        "Il gatto rincorre il topo",                                     # Breve
        "Oggi il cielo è molto limpido e azzurro sopra le montagne",    # Lunga
        "Deep Learning è fantastico",                                    # Corta
        "La memoria delle reti neurali è complessa ma affascinante"     # Media
    ]
    # Etichette di target: vogliamo che la rete impari a classificarle (es. Sentiment Positivo = 1)
    etichette = np.array([1, 1, 1, 1], dtype="float32")
    
    return testi, etichette

def preprocessamento_dati(testi):
    """
    FASE 2: TRADUZIONE UMANO -> MACCHINA.
    I computer non capiscono le lettere, leggono solo numeri (tensori).
    """
    # 1. Inizializziamo il Vectorizer: agisce come un dizionario dinamico.
    # max_tokens=100: tiene solo le 100 parole più comuni.
    # output_sequence_length=12: taglia le frasi lunghe o allunga quelle corte 
    # aggiungendo zeri (PADDING) per rendere ogni "striscia" di dati lunga 12.
    vectorizer = layers.TextVectorization(max_tokens=100, output_sequence_length=12)
    
    # 2. ADAPT: Il vectorizer legge tutti i testi per imparare quali parole esistono.
    # Costruisce internamente la mappa: "gatto" -> 5, "cielo" -> 12, ecc.
    vectorizer.adapt(testi)
    
    # 3. TRASFORMAZIONE: Applichiamo la mappa. Ogni frase diventa una lista di 12 numeri.
    # Gli zeri finali che vedrai sono il "silenzio" (padding) per pareggiare le lunghezze.
    sequenze = vectorizer(testi)
    
    return sequenze, vectorizer

def build_modern_rnn_model(vocab_size):
    """
    FASE 3: COSTRUZIONE DEL CERVELLO (MODELLO).
    Qui definiamo come le informazioni fluiscono attraverso i neuroni.
    """
    # --- [A] PORTA D'INGRESSO (INPUT) ---
    # Specifichiamo che riceveremo sequenze di 12 numeri interi.
    inputs = keras.Input(shape=(12,), name="Input_ID_Parole")

    # --- [B] TRADUTTORE DI CONCETTI (EMBEDDING) ---
    # Trasforma ogni numero (es. 5) in un vettore denso (16 numeri decimali).
    # Parole simili avranno vettori simili nello spazio matematico.
    # mask_zero=True: FONDAMENTALE. Dice alla rete: "Se vedi uno zero, è solo 
    # padding per pareggiare la lunghezza, ignoralo nei calcoli della memoria!"
    x = layers.Embedding(input_dim=vocab_size, output_dim=16, mask_zero=True)(inputs)

    # --- [C] IL CUORE DELLA MEMORIA (LSTM) ---
    # Una SimpleRNN dimentica velocemente l'inizio della frase.
    # La LSTM (Long Short-Term Memory) usa dei "gate" (cancelli) per decidere 
    # cosa ricordare e cosa dimenticare del passato. 
    # 32 sono i "neuroni" o unità di memoria interna.
    x = layers.LSTM(32, name="Memoria_Contestuale")(x)

    # --- [D] IL DECISORE FINALE (DENSE) ---
    # Prende il riassunto fatto dalla LSTM e sputa fuori un unico numero.
    # activation="sigmoid": schiaccia il risultato tra 0 e 1 (probabilità).
    outputs = layers.Dense(1, activation="sigmoid", name="Probabilita_Sentiment")(x)

    # --- [E] ASSEMBLAGGIO ---
    model = keras.Model(inputs=inputs, outputs=outputs, name="Rete_Neurale_Sequenziale")
    
    # --- [F] COMPILAZIONE (STRATEGIA DI STUDIO) ---
    # optimizer="adam": l'algoritmo che corregge gli errori durante lo studio.
    # loss="binary_crossentropy": come calcoliamo quanto la rete ha sbagliato.
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    
    return model

# ================================================================================
# --- CICLO DI ESECUZIONE (LOGICA FLUSSO LAVORO) ---
# ================================================================================

# 1. Recuperiamo i testi originali
testi_raw, etichette = crea_dataset_didattico()

# 2. Trasformiamo i testi in matrici numeriche leggibili dalla GPU
x_train, vect_layer = preprocessamento_dati(testi_raw)

# 3. Curiosità: vediamo come la macchina ha "mappato" le parole
vocabulario = vect_layer.get_vocabulary()
print(f"\n[INFO] Vocabolario Identificato: {vocabulario}")
print(f"[INFO] Esempio Frase 1 (Vettorizzata con Padding):\n{x_train[0]}")

# 4. Creiamo il cervello basandoci sulla dimensione del vocabolario trovato
model = build_modern_rnn_model(len(vocabulario))

# 5. Visualizziamo la struttura interna: vedrai come il "Param #" (numeri da imparare) 
# si sposta dall'Embedding alla LSTM.
print("\n[MAPPA DEL MODELLO]")
model.summary()

# Ora il modello è pronto per essere addestrato con: model.fit(x_train, etichette, epochs=...)


[INFO] Vocabolario Identificato: ['', '[UNK]', np.str_('è'), np.str_('il'), np.str_('topo'), np.str_('sopra'), np.str_('rincorre'), np.str_('reti'), np.str_('oggi'), np.str_('neurali'), np.str_('montagne'), np.str_('molto'), np.str_('memoria'), np.str_('ma'), np.str_('limpido'), np.str_('learning'), np.str_('le'), np.str_('la'), np.str_('gatto'), np.str_('fantastico'), np.str_('e'), np.str_('delle'), np.str_('deep'), np.str_('complessa'), np.str_('cielo'), np.str_('azzurro'), np.str_('affascinante')]
[INFO] Esempio Frase 1 (Vettorizzata con Padding):
tensor([ 3, 18,  6,  3,  4,  0,  0,  0,  0,  0,  0,  0], device='cuda:0')

[MAPPA DEL MODELLO]


Model: "Rete_Neurale_Sequenziale"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Input_ID_Parole     │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 12, 16)    │        432 │ Input_ID_Parole[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 12)        │          0 │ Input_ID_Parole[… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Memoria_Contestuale │ (None, 32)        │      6,272 │ embedding[0][0],  │
│ (LSTM)              │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Probabilita_Sentim… │ (None, 1)         │         33 │ Memoria_Contestu… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,737 (26.32 KB)

 Trainable params: 6,737 (26.32 KB)

 Non-trainable params: 0 (0.00 B)